# 可选实验：逻辑回归的代价函数

## 目标
在本实验中，你将：
- 检查并运用逻辑回归的代价函数。

In [1]:
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
plt.rcParams['font.size'] = 8
from lab_utils_common import  plot_data, sigmoid, dlc

## 数据集
让我们从决策边界实验中所用的同一个数据集开始。

In [2]:
import numpy as np

X = np.array([[0.5, 1.5], [1,1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y = np.array([0, 0, 0, 1, 1, 1])

我们将用一个辅助函数来绘制这些数据。标签为 $y=1$ 的数据点显示为红色叉号，标签为 $y=0$ 的数据点显示为蓝色圆圈。

In [3]:
fig,ax = plt.subplots(1,1,figsize=(4,4))
plot_data(X, y, ax)

# 把两个坐标轴都设为 0-4
ax.axis([0, 4, 0, 3.5])
ax.set_ylabel('$x_1$', fontsize=12)
ax.set_xlabel('$x_0$', fontsize=12)
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## 代价函数

在之前的实验中，你推导了*逻辑损失*函数。回忆一下，损失定义为作用于单个样本。这里你把各个损失组合起来构成**代价**，它涵盖所有样本。


回忆一下，对于逻辑回归，代价函数的形式为

$$ J(\\mathbf{w},b) = \\frac{1}{m} \\sum_{i=0}^{m-1} \\left[ loss(f_{\\mathbf{w},b}(\\mathbf{x}^{(i)}), y^{(i)}) \\right] \\tag{1}$$

其中
* $loss(f_{\\mathbf{w},b}(\\mathbf{x}^{(i)}), y^{(i)})$ 是单个数据点的代价，即：

    $$loss(f_{\\mathbf{w},b}(\\mathbf{x}^{(i)}), y^{(i)}) = -y^{(i)} \\log\\left(f_{\\mathbf{w},b}\\left( \\mathbf{x}^{(i)} \\right) \\right) - \\left( 1 - y^{(i)}\\right) \\log \\left( 1 - f_{\\mathbf{w},b}\\left( \\mathbf{x}^{(i)} \\right) \\right) \\tag{2}$$
    
*  其中 m 是数据集中的训练样本数，并且：
$$
\\begin{align}
  f_{\\mathbf{w},b}(\\mathbf{x^{(i)}}) &= g(z^{(i)})\\tag{3} \\\\
  z^{(i)} &= \\mathbf{w} \\cdot \\mathbf{x}+ b\\tag{4} \\\\
  g(z^{(i)}) &= \\frac{1}{1+e^{-z^{(i)}}}\\tag{5} 
\\end{align}
$$
 

<a name='ex-02'></a>
#### 代码说明

`compute_cost_logistic` 的算法如下：
- 在循环外创建一个变量来存放代价
- 遍历训练集中的所有样本。
    - 计算每个训练样本的损失
        - 计算 `z_i`，即上面的公式 (4)
        - 预测 `f_wb_i`，其中 `g` 是 sigmoid 函数，即上面的公式 (3)。Sigmoid（公式 5）是一个库函数。
        - 计算该样本的损失，即上面的公式 (2)
     - 把这个代价加到循环外创建的总代价变量上
- 把所有迭代的代价求和，并返回总和除以样本数。


注意，变量 X_train 和 y_train 不是标量值，而是形状分别为 ($m, n$) 和 ($𝑚$,) 的矩阵，其中 $𝑛$ 是特征数，$𝑚$ 是训练样本数。


In [4]:
def compute_cost_logistic(X, y, w, b):
    """计算代价

    参数：
      X (ndarray): 形状 (m,n)，含 n 个特征的样本矩阵
      y (ndarray): 形状 (m,)  目标值
      w (ndarray): 形状 (n)   用于预测的参数
      b (scalar):              用于预测的参数
     
    返回：
      cost (scalar): 代价"""

    m = X.shape[0]
    cost = 0.0
    for i in range(m):
        z_i = np.dot(X[i],w) + b
        f_wb_i = sigmoid(z_i)
        cost +=  -y[i]*np.log(f_wb_i) - (1-y[i])*np.log(1-f_wb_i)
             
    cost = (1/m) * cost
    return cost


用下面的单元格检查代价函数的实现。

In [5]:
# 注意，w 定义为列向量
w = np.array([1,1])
b = -3
print(compute_cost_logistic(X, y.reshape(-1), w.reshape(-1), b))

0.3668667864055174


**预期输出**：0.3668667864055174

## 示例
现在，让我们看看 $w$ 取另一个值时代价函数的输出。

* 在之前的实验中，你为 $b = -3, w_0 = 1, w_1 = 1$ 绘制了决策边界。也就是说，你的 `w = np.array([-3,1,1])`。

* 假设你想看看 $b = -4, w_0 = 1, w_1 = 1$，即 `w = np.array([-4,1,1])` 是否给出更好的模型。

让我们先为这两个不同的 $b$ 值绘制决策边界，看看哪一个更好地拟合数据。

* 对于 $b = -3, w_0 = 1, w_1 = 1$，我们画 $-3 + x_0+x_1 = 0$（蓝色所示）
* 对于 $b = -4, w_0 = 1, w_1 = 1$，我们画 $-4 + x_0+x_1 = 0$（洋红色所示）

In [6]:
import matplotlib.pyplot as plt

# 在 0 到 6 之间取值
x0 = np.arange(0,6)

# 绘制两条决策边界
x1 = 3 - x0
x1_other = 4 - x0

fig,ax = plt.subplots(1, 1, figsize=(4,4))
# 绘制决策边界
ax.plot(x0,x1, c=dlc["dlblue"], label="$b$=-3")
ax.plot(x0,x1_other, c=dlc["dlmagenta"], label="$b$=-4")
ax.axis([0, 4, 0, 4])

# 绘制原始数据
plot_data(X,y,ax)
ax.axis([0, 4, 0, 4])
ax.set_ylabel('$x_1$', fontsize=12)
ax.set_xlabel('$x_0$', fontsize=12)
plt.legend(loc="upper right")
plt.title("Decision Boundary")
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

从图中可以看出，`w = np.array([-4,1,1])` 对训练数据来说是更差的模型。让我们看看代价函数的实现是否反映了这一点。

In [7]:
w_array1 = np.array([1,1])
b_1 = -3
w_array2 = np.array([1,1])
b_2 = -4

print("Cost for b = -3 : ", compute_cost_logistic(X, y, w_array1, b_1))
print("Cost for b = -4 : ", compute_cost_logistic(X, y, w_array2, b_2))

Cost for b = -3 :  0.3668667864055174
Cost for b = -4 :  0.503680863674846


**预期输出**

b = -3 时的代价： 0.3668667864055174

b = -4 时的代价： 0.503680863674846


你可以看到代价函数的表现符合预期，`w = np.array([-4,1,1])` 的代价确实高于 `w = np.array([-3,1,1])` 的代价

## 恭喜！
在本实验中，你检查并运用了逻辑回归的代价函数。